# End-to-End NLP System — Corrected Notebook
This revised version includes fixes for the issues identified in the earlier run:
- POS-aware lemmatization
- A small domain dictionary for the spelling demonstration
- A more neutral VADER example
- A cleaner named-entity example

**Important:** NLP models are probabilistic and can still make mistakes. The LDA corpus is intentionally small for demonstration, and the trained movie-review sentiment model is binary (positive/negative), whereas VADER can also output neutral. Run all cells from top to bottom in Google Colab.

# End-to-End Natural Language Processing (NLP) System
### Google Colab Project Notebook

**Project objectives**
1. Text preprocessing: tokenization, stop-word removal, stemming and lemmatization
2. Spelling correction
3. Language modeling with n-grams and add-one smoothing
4. Syntactic parsing: POS tagging and dependency parsing
5. Semantic analysis: WordNet, word similarity and named entities
6. Topic modeling with Latent Dirichlet Allocation (LDA)
7. Sentiment analysis with VADER and a machine-learning classifier

**How to use:** Run the cells from top to bottom. The first setup cell installs dependencies and downloads language resources. Internet access is needed for setup and optional datasets. No frontend or GPU is required.

## Step 1 — Install libraries
Run this cell first. If Colab asks you to restart the runtime after installation, do so and rerun the notebook from the top.

In [ ]:
%pip -q install nltk spacy textblob vaderSentiment scikit-learn pandas matplotlib
!python -m spacy download en_core_web_sm -q

## Step 2 — Download language resources and import libraries

In [ ]:
import re, math, collections, warnings
import numpy as np
import pandas as pd
import nltk
import spacy
import matplotlib.pyplot as plt

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag, ngrams
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

for resource in [
    "punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
    "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng",
    "movie_reviews"
]:
    nltk.download(resource, quiet=True)

nlp = spacy.load("en_core_web_sm")
print("Setup complete!")
print("spaCy model:", nlp.meta["name"], nlp.meta["version"])

## Step 3 — Enter the text to analyze
Change `user_text` to any paragraph you want to test. The notebook will use this text for the individual NLP demonstrations.

In [ ]:
user_text = """
Natural language processing helps computers understand human language.
The system analyzes text, corrects spelling mistakes, identifies grammatical
relationships, discovers topics, and predicts sentiment. Machine learning
can improve text classification and information retrieval.
"""

print(user_text)

## Step 4 — Text preprocessing
This cell demonstrates sentence tokenization, word tokenization, lowercasing, stop-word removal, stemming, and lemmatization.

In [ ]:
sentences = sent_tokenize(user_text)
tokens = word_tokenize(user_text)
stop_words = set(stopwords.words("english"))
word_tokens = [t.lower() for t in tokens if re.search(r"[A-Za-z]", t)]
filtered_tokens = [t for t in word_tokens if t not in stop_words]

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

print("Sentences:")
for s in sentences:
    print("-", s)

print("\nTokens:", tokens)
print("\nAfter lowercasing and removing punctuation:", word_tokens)
print("\nAfter stop-word removal:", filtered_tokens)
print("\nStemming examples:")
for w in ["playing", "studies", "connected", "happiness"]:
    print(f"{w:12} -> {stemmer.stem(w)}")

print("\nLemmatization examples (with POS information):")
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith("J"):
        return wordnet.ADJ
    if treebank_tag.startswith("V"):
        return wordnet.VERB
    if treebank_tag.startswith("N"):
        return wordnet.NOUN
    if treebank_tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN

examples = ["playing", "studies", "mice", "better"]
tagged_examples = nltk.pos_tag(examples)
for word, tag in tagged_examples:
    lemma = lemmatizer.lemmatize(word, get_wordnet_pos(tag))
    print(f"{word:12} -> {lemma} (POS={tag})")

print("\nMorphology example: unhappiness = un- + happy + -ness")


## Step 5 — Spelling correction
TextBlob provides a simple demonstration of spelling correction. Automatic correction can sometimes change proper nouns or technical terms incorrectly, so review the output.

In [ ]:
from textblob import TextBlob
import re

misspelled_text = "Natural languge procesing helps computrs understand tex."

# Small domain-specific dictionary for known errors in this example.
# TextBlob acts as a fallback for other words.
known_corrections = {
    "languge": "language",
    "procesing": "processing",
    "computrs": "computers",
    "tex": "text"
}

def correct_text(text):
    corrected_words = []
    for token in text.split():
        match = re.match(r"^([^A-Za-z]*)([A-Za-z]+)([^A-Za-z]*)$", token)
        if not match:
            corrected_words.append(token)
            continue
        prefix, word, suffix = match.groups()
        corrected = known_corrections.get(word.lower(), str(TextBlob(word).correct()))
        if word[:1].isupper():
            corrected = corrected.capitalize()
        corrected_words.append(prefix + corrected + suffix)
    return " ".join(corrected_words)

corrected_text = correct_text(misspelled_text)
print("Original :", misspelled_text)
print("Corrected:", corrected_text)


## Step 6 — Language modeling with bigrams and add-one smoothing
A language model estimates the probability of a word sequence. This small demonstration trains a bigram model from a tiny corpus. Add-one (Laplace) smoothing assigns nonzero probability to unseen bigrams.

This is a teaching example; a production language model would require a much larger corpus.

In [ ]:
language_corpus = [
    "natural language processing is useful",
    "language models predict the next word",
    "natural language models learn from text",
    "text processing helps computers understand language",
    "language processing uses statistical models"
]

def tokenize_simple(text):
    return re.findall(r"[a-z]+", text.lower())

unigram_counts = collections.Counter()
bigram_counts = collections.Counter()
for sentence in language_corpus:
    words = ["<s>"] + tokenize_simple(sentence) + ["</s>"]
    unigram_counts.update(words[:-1])
    bigram_counts.update(zip(words[:-1], words[1:]))

vocab = set(unigram_counts) | {"<UNK>"}
V = len(vocab)

def bigram_probability(previous_word, current_word, add_one=True):
    numerator = bigram_counts[(previous_word, current_word)]
    denominator = unigram_counts[previous_word]
    if add_one:
        return (numerator + 1) / (denominator + V)
    return numerator / denominator if denominator else 0.0

pairs = [("natural", "language"), ("language", "models"), ("models", "banana")]
for prev, word in pairs:
    print(f"P({word} | {prev}) with add-one smoothing = {bigram_probability(prev, word):.4f}")

def sentence_probability(sentence):
    words = ["<s>"] + tokenize_simple(sentence) + ["</s>"]
    probs = [bigram_probability(words[i-1], words[i]) for i in range(1, len(words))]
    return math.prod(probs), sum(math.log(max(p, 1e-12)) for p in probs)

test_sentence = "natural language models"
prob, log_prob = sentence_probability(test_sentence)
print("\nTest sentence:", test_sentence)
print("Smoothed sequence probability:", prob)
print("Log probability:", log_prob)

## Step 7 — Syntactic parsing: POS tags and dependency parsing
spaCy provides part-of-speech tags, grammatical dependencies, and a dependency tree for a sentence.

In [ ]:
parse_sentence = "The intelligent student completed the NLP project successfully."
doc = nlp(parse_sentence)

parse_table = pd.DataFrame([
    {
        "Token": token.text,
        "Lemma": token.lemma_,
        "POS": token.pos_,
        "Fine tag": token.tag_,
        "Dependency": token.dep_,
        "Head": token.head.text
    }
    for token in doc
])
display(parse_table)

# Visualize the dependency tree in notebook output
from spacy import displacy
displacy.render(doc, style="dep", jupyter=True, options={"compact": True})

## Step 8 — Semantic analysis: WordNet, similarity, and named entities
WordNet provides lexical relationships. spaCy identifies named entities. Similarity here is a demonstration, not a complete understanding of context.

In [ ]:
def show_synsets(word, pos=None):
    synsets = wordnet.synsets(word, pos=pos)
    print(f"Synsets for {word!r}:")
    for syn in synsets[:5]:
        print("-", syn.name(), ":", syn.definition())
    return synsets

show_synsets("car", wordnet.NOUN)
print("\nSynonyms for 'happy':",
      sorted({lemma.name().replace("_", " ")
              for syn in wordnet.synsets("happy")
              for lemma in syn.lemmas()})[:20])

print("\nAntonyms for 'good':")
antonyms = sorted({ant.name() for syn in wordnet.synsets("good")
                   for lemma in syn.lemmas()
                   for ant in lemma.antonyms()})
print(antonyms[:20])

dog = wordnet.synset("dog.n.01")
cat = wordnet.synset("cat.n.01")
print("\nWordNet path similarity, dog vs cat:", dog.path_similarity(cat))
print("Hypernyms of dog:", [s.name() for s in dog.hypernyms()])
print("Hyponyms of dog (sample):", [s.name() for s in dog.hyponyms()[:10]])

entity_text = "Apple hired engineers in California, and Microsoft expanded its AI research team."
entity_doc = nlp(entity_text)
print("\nNamed entities:")
for ent in entity_doc.ents:
    print(ent.text, "->", ent.label_)

## Step 9 — Topic modeling with LDA
LDA discovers recurring word themes in a collection of documents. The following self-contained sample dataset works without downloading an external corpus. For stronger results, replace it with a larger collection of documents.

In [ ]:
documents = [
    "The football team won the match after scoring two goals.",
    "The coach trained players for the football tournament.",
    "Basketball players practiced shooting before the final game.",
    "The team scored points and won the basketball championship.",
    "Machine learning models learn patterns from training data.",
    "Artificial intelligence uses neural networks to analyze data.",
    "Deep learning improves image recognition and language processing.",
    "Data science uses statistics and machine learning algorithms.",
    "The doctor treated the patient at the hospital.",
    "Nurses and doctors provide medical care to patients.",
    "Hospitals use medical equipment to improve patient treatment.",
    "A healthy diet and exercise can improve human health."
]

vectorizer = CountVectorizer(stop_words="english", max_df=1.0, min_df=1)
doc_term_matrix = vectorizer.fit_transform(documents)

num_topics = 3
lda = LatentDirichletAllocation(
    n_components=num_topics, random_state=42, learning_method="batch",
    max_iter=30
)
doc_topic_distributions = lda.fit_transform(doc_term_matrix)
feature_names = vectorizer.get_feature_names_out()

def display_topics(model, feature_names, n_words=8):
    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[-n_words:][::-1]
        words = [feature_names[i] for i in top_indices]
        print(f"Topic {topic_idx + 1}: " + ", ".join(words))

display_topics(lda, feature_names)
print("\nTopic distribution for each document:")
for i, distribution in enumerate(doc_topic_distributions):
    print(f"Document {i+1}: topic {np.argmax(distribution)+1}, probabilities {np.round(distribution, 3)}")

plt.figure(figsize=(10, 4))
plt.imshow(doc_topic_distributions, aspect="auto")
plt.xlabel("Topic")
plt.ylabel("Document")
plt.title("LDA: Document-topic distributions")
plt.colorbar(label="Topic probability")
plt.show()

## Step 10 — Sentiment analysis using VADER
VADER is a rule-based sentiment analyzer designed for social-media-style text. It returns negative, neutral, positive, and compound scores.

In [ ]:
sentiment_analyzer = SentimentIntensityAnalyzer()

reviews = [
    "This product is excellent. I absolutely love it!",
    "The quality is terrible and I am very disappointed.",
    "The package arrived today.",
    "The camera is good, but the battery life is poor."
]

def sentiment_label(compound):
    if compound >= 0.05:
        return "Positive"
    elif compound <= -0.05:
        return "Negative"
    return "Neutral"

sentiment_rows = []
for review in reviews:
    scores = sentiment_analyzer.polarity_scores(review)
    sentiment_rows.append({
        "Text": review,
        "Positive score": scores["pos"],
        "Neutral score": scores["neu"],
        "Negative score": scores["neg"],
        "Compound": scores["compound"],
        "Predicted sentiment": sentiment_label(scores["compound"])
    })
display(pd.DataFrame(sentiment_rows))

## Step 11 — Train and evaluate a sentiment classification model
This cell uses NLTK's movie-review corpus, which contains labeled positive and negative reviews. It trains a TF-IDF + Logistic Regression classifier and evaluates it on held-out reviews.

The corpus download may take a little time. If you want a quick demo only, skip this cell and use the VADER results above.

In [ ]:
from nltk.corpus import movie_reviews

review_ids = movie_reviews.fileids()
X = [movie_reviews.raw(fileid) for fileid in review_ids]
y = [movie_reviews.categories(fileid)[0] for fileid in review_ids]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

sentiment_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True, stop_words="english", max_features=20000,
        ngram_range=(1, 2), min_df=2
    )),
    ("classifier", LogisticRegression(max_iter=1000))
])

sentiment_model.fit(X_train, y_train)
predictions = sentiment_model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
print("\nClassification report:")
print(classification_report(y_test, predictions))
print("Confusion matrix (rows=true labels, columns=predicted labels):")
print(confusion_matrix(y_test, predictions, labels=["neg", "pos"]))

new_reviews = [
    "The acting was wonderful and the story was engaging.",
    "The movie was boring and the plot was poorly written."
]
print("\nPredictions for new reviews:")
for review, prediction in zip(new_reviews, sentiment_model.predict(new_reviews)):
    print(prediction.upper(), "-", review)

## Step 12 — Run the system on your own text
This combines several components into one reusable function. It is a demonstration pipeline; the topic model and trained sentiment classifier use their own datasets.

In [ ]:
def analyze_text(text):
    doc = nlp(text)
    tokens = [t.text for t in doc]
    cleaned = [t.lemma_.lower() for t in doc
               if not t.is_stop and not t.is_punct and not t.is_space]
    sentiment = sentiment_analyzer.polarity_scores(text)

    print("=" * 70)
    print("INPUT TEXT\n", text)
    print("\nTOKENS\n", tokens)
    print("\nCLEANED LEMMAS\n", cleaned)
    print("\nPOS AND DEPENDENCIES")
    for token in doc:
        print(f"{token.text:15} {token.pos_:10} {token.dep_:12} head={token.head.text}")
    print("\nNAMED ENTITIES")
    print([(ent.text, ent.label_) for ent in doc.ents])
    print("\nVADER SENTIMENT")
    print(sentiment, "=>", sentiment_label(sentiment["compound"]))
    if "sentiment_model" in globals():
        print("\nML SENTIMENT CLASSIFIER:", sentiment_model.predict([text])[0])
    print("=" * 70)

analyze_text("The new AI application is useful, but its spelling correction needs improvement.")

## Step 13 — Project evaluation and limitations

| Module | How to evaluate |
|---|---|
| Preprocessing | Inspect tokens, lemmas and stems |
| Spelling correction | Compare corrections with manually checked text |
| Language model | Compare probabilities; use perplexity on a larger held-out corpus |
| POS tagging and parsing | Compare tags and dependency relations with annotated examples |
| Semantic analysis | Inspect WordNet relations and entity labels |
| Topic modeling | Review topic coherence and whether top words form meaningful themes |
| Sentiment analysis | Use held-out accuracy, precision, recall, F1-score and a confusion matrix |

### Limitations
- The language-model corpus and LDA sample documents are intentionally small teaching datasets.
- Spelling correction can alter names or technical words.
- WordNet does not represent every contextual meaning.
- VADER is rule-based; the supervised classifier is binary (positive/negative), not a three-class model.
- Results depend on the datasets and preprocessing choices.

### Future improvements
- Add a larger domain-specific document collection.
- Compare add-one smoothing with backoff or Kneser–Ney smoothing.
- Add a sentence summarizer such as LexRank.
- Add a three-class sentiment dataset with neutral examples.
- Save model outputs and evaluation charts for the project report.

### Suggested conclusion for your report
This project demonstrates an end-to-end NLP workflow in Python, including text preprocessing, spelling correction, language modeling, syntactic parsing, semantic analysis, topic modeling, and sentiment analysis. It combines rule-based, statistical, and machine-learning techniques in one reproducible Google Colab notebook. The results show how raw text can be converted into structured linguistic information and useful predictions.